# B2-023-generative-models-diffusion — Practice p10 — Solution

**Type:** constrained-coding · **Difficulty:** core · **Concepts:** denoising-diffusion-probabilistic-models

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

## Supplied cell (copied verbatim from the statement)

In [ ]:
import torch
from torch import nn

SEED = 20261015
X0 = torch.tensor([[1.0, -2.0], [0.5, 0.5], [-1.0, 3.0]])
T_IDX = torch.tensor([1, 3, 5])
EPS = torch.tensor([[0.5, 0.0], [-1.0, 2.0], [0.2, -0.4]])


class ToyEps(nn.Module):
    def forward(self, x_t, t_scaled):
        return 0.5 * x_t + t_scaled


class ZeroEps(nn.Module):
    def forward(self, x_t, t_scaled):
        return torch.zeros_like(x_t)


ATOL = 1e-6
RTOL = 1e-6

## Solution

`ddpm_loss` calls the module-level `q_sample` by name (so a test can wrap it), builds the scaled time feature `t_scaled = (t.to(x0.dtype) / T).unsqueeze(1)` of shape `(B,1)`, asks the model for `eps_hat`, checks its shape, and returns the explicit mean `((eps_hat - eps) ** 2).mean()` over all $B\times d$ entries.
No random numbers are drawn inside; `t` and `eps` are arguments.

In [ ]:
import math


def make_schedule(T, beta_1, beta_T):
    if not isinstance(T, int) or isinstance(T, bool) or T < 2:
        raise ValueError("T must be an int >= 2")
    if not (0 < beta_1 <= beta_T < 1):
        raise ValueError("need 0 < beta_1 <= beta_T < 1")
    beta = torch.zeros(T + 1, dtype=torch.float64)
    beta[1:] = torch.linspace(beta_1, beta_T, T, dtype=torch.float64)
    alpha = 1 - beta
    alpha_bar = torch.cumprod(alpha, dim=0)
    beta_tilde = torch.zeros(T + 1, dtype=torch.float64)
    beta_tilde[1:] = (1 - alpha_bar[:-1]) / (1 - alpha_bar[1:]) * beta[1:]
    return {"beta": beta, "alpha": alpha, "alpha_bar": alpha_bar, "beta_tilde": beta_tilde}


def q_sample(x0, t, eps, alpha_bar):
    if x0.shape != eps.shape:
        raise ValueError("x0 and eps must have the same shape")
    if (not isinstance(t, torch.Tensor) or t.dtype != torch.int64 or t.dim() != 1
            or x0.dim() != 2 or t.shape[0] != x0.shape[0]):
        raise ValueError("t must be an int64 tensor of shape (B,)")
    T = alpha_bar.shape[0] - 1
    if bool((t < 1).any()) or bool((t > T).any()):
        raise ValueError("timesteps must lie in 1..T")
    ab = alpha_bar[t].to(x0.dtype).unsqueeze(1)
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps


def ddpm_loss(model, x0, t, eps, alpha_bar):
    T = alpha_bar.shape[0] - 1
    x_t = q_sample(x0, t, eps, alpha_bar)
    t_scaled = (t.to(x0.dtype) / T).unsqueeze(1)
    eps_hat = model(x_t, t_scaled)
    if eps_hat.shape != eps.shape:
        raise ValueError("model output must have the shape of eps")
    return ((eps_hat - eps) ** 2).mean()


class Denoiser(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2))

    def forward(self, x_t, t_scaled):
        return self.net(torch.cat([x_t, t_scaled], dim=1))


schedule = make_schedule(5, 0.1, 0.5)

### Probes

In [ ]:
loss_toy = ddpm_loss(ToyEps(), X0, T_IDX, EPS, schedule["alpha_bar"])
loss_zero = ddpm_loss(ZeroEps(), X0, T_IDX, EPS, schedule["alpha_bar"])


class Spy(nn.Module):
    def __init__(self):
        super().__init__()
        self.seen = []

    def forward(self, x_t, t_scaled):
        self.seen.append((x_t.detach().clone(), t_scaled.detach().clone()))
        return torch.zeros_like(x_t)


spy = Spy()
ddpm_loss(spy, X0, T_IDX, EPS, schedule["alpha_bar"])
spy_x_t, spy_t = spy.seen[0]


def ddpm_loss_x0_target(model, x0, t, eps, alpha_bar):
    T = alpha_bar.shape[0] - 1
    x_t = q_sample(x0, t, eps, alpha_bar)
    eps_hat = model(x_t, (t.to(x0.dtype) / T).unsqueeze(1))
    return ((eps_hat - x0) ** 2).mean()


def ddpm_loss_sum(model, x0, t, eps, alpha_bar):
    T = alpha_bar.shape[0] - 1
    x_t = q_sample(x0, t, eps, alpha_bar)
    eps_hat = model(x_t, (t.to(x0.dtype) / T).unsqueeze(1))
    return ((eps_hat - eps) ** 2).sum()


loss_x0_variant = ddpm_loss_x0_target(ToyEps(), X0, T_IDX, EPS, schedule["alpha_bar"])
loss_sum_variant = ddpm_loss_sum(ToyEps(), X0, T_IDX, EPS, schedule["alpha_bar"])

torch.manual_seed(SEED)
model = Denoiser()
loss_net = ddpm_loss(model, X0, T_IDX, EPS, schedule["alpha_bar"])
loss_net.backward()
grads = [p.grad for p in model.parameters()]


class BadShape(nn.Module):
    def forward(self, x_t, t_scaled):
        return x_t[:, :1]


try:
    ddpm_loss(BadShape(), X0, T_IDX, EPS, schedule["alpha_bar"])
    bad_shape_raised = False
except ValueError:
    bad_shape_raised = True

print(f"probe 1: {loss_toy.item():.10f}   probe 2: {loss_zero.item():.7f}")
print("probe 3:", tuple(spy_t.shape), spy_t.dtype, spy_t.flatten().tolist())
print(f"probe 4: x0-target {loss_x0_variant.item():.10f}  sum {loss_sum_variant.item():.6f}")
print(f"probe 5: loss {loss_net.item():.6f}  shape {tuple(loss_net.shape)}")
print("probe 6:", bad_shape_raised)

### Why the reduction must be stated

A sum over entries equals $B\cdot d$ times the mean, so the loss value, every threshold stated against it, and the effective step size of the optimizer all change by a factor that depends on the batch and data dimension.
Unless the contract pins the reduction, two correct-looking implementations produce numbers that differ by that factor, and no certificate ("loss below 0.40", "decrease to 60%") is reproducible across them.

### Answer check

In [ ]:
# Probe 1
assert torch.allclose(loss_toy, torch.tensor(1.1080242395), atol=ATOL, rtol=RTOL)
# Probe 2
assert torch.allclose(loss_zero, EPS.pow(2).mean(), atol=ATOL, rtol=RTOL)
assert abs(loss_zero.item() - 5.45 / 6) < 1e-6
# Probe 3
assert tuple(spy_t.shape) == (3, 1) and spy_t.dtype == torch.float32
assert torch.equal(spy_t, torch.tensor([[0.2], [0.6], [1.0]]))
assert torch.equal(spy_x_t, q_sample(X0, T_IDX, EPS, schedule["alpha_bar"]))
# Probe 4
assert abs(loss_x0_variant.item() - 1.4600834846) < 1e-6
assert abs(loss_x0_variant.item() - loss_toy.item()) > 0.1
assert abs(loss_sum_variant.item() - loss_toy.item()) > 0.1
# Probe 5
assert loss_net.shape == ()
assert all(g is not None and bool(torch.isfinite(g).all()) for g in grads)
assert any(bool((g != 0).any()) for g in grads)
# Probe 6
assert bad_shape_raised